In [18]:
import sympy as sm
import sympy.physics.mechanics as me
me.init_vprinting()

In [ ]:
#Symbols
phi, ksi, theta, delta, gamma, beta, psi = me.dynamicsymbols('φ xi θ δ γ β ψ') #angles
R, l1, l2, r1, r2, r3, r_rope = sm.symbols("R, l1, l2, r1, r2, r3, r_rope") #lengths and wheel radii
m1, m2, m3, m_rod, m_rope, g= sm.symbols("m1, m2, m3, m_rod, m_rope, g")
t = me.dynamicsymbols._t

#Reference Frames
N = me.ReferenceFrame('N') #fixed ground, lowest possible point on the rope
A = me.ReferenceFrame('A') #point on the rope where the wheel contact is
B1 = me.ReferenceFrame('B1') #rolling wheel, leaning only
B2 = me.ReferenceFrame('B2') #rolling wheel, leaning and heading
B3 = me.ReferenceFrame('B3') #rolling wheel, leading, heading, and rolling
C = me.ReferenceFrame('C') #Body frame
D = me.ReferenceFrame('D') #Balancing wheel
E = me.ReferenceFrame('E') #Steering wheel

#Orienting Reference Frames
A.orient_axis(N, phi, N.x)
B1.orient_axis(A, ksi, A.z) #heading/steering
B2.orient_axis(B1, delta, B1.x) #tilt
B3.orient_axis(B2, theta, B2.y) #rolling
C.orient_axis(B1, gamma, B1.y) #body frame, defined only by the heading and leaning of the rolling wheel, not the rolling
D.orient_axis(C, beta, C.x)   
E.orient_axis(C, psi, C.z)

#Points
O = me.Point('O')
PA = me.Point('PA')
PB12 = me.Point('PB12')
PB3 = me.Point('PB3')
PC = me.Point('PC')
PD = me.Point('PD')
PE = me.Point('PE')

#Positions of points
O.set_vel(N, 0)
PA.set_pos(O, (R*sm.sin(phi)*N.y + R*(1-sm.cos(phi))*N.z))
PB12.set_pos(PA, 0)
PB3.set_pos(PB12, r1* B2.z)
PC.set_pos(PB12, l1*B2.z)
PD.set_pos(PB12, (l1+l2)*B2.z)
PE.set_pos(PB12, (l1+ l2)*B2.z)

#Positions from the origin point
O_r_PA = PA.pos_from(O).express(N)
O_r_PB12 = PB12.pos_from(O).express(N)
O_r_PB3 = PB3.pos_from(O).express(N)
O_r_PC = PC.pos_from(O).express(N)
O_r_PD = PD.pos_from(O).express(N)
O_r_PE = PE.pos_from(O).express(N)

#Differentiated vectors
O_r_PA_dot = O_r_PA.dt(N)
O_r_PB12_dot = O_r_PB12.dt(N)
O_r_PB3_dot = O_r_PB3.dt(N)
O_r_PC_dot = O_r_PC.dt(N)
O_r_PD_dot = O_r_PD.dt(N)
O_r_PE_dot = O_r_PE.dt(N)

#Squared Magnitude of Vectors

PA_squared = O_r_PA_dot.dot(O_r_PA_dot)
PB12_squared = O_r_PB12_dot.dot(O_r_PB12_dot)
PB3_squared = O_r_PB3_dot.dot(O_r_PB3_dot)
PC_squared = O_r_PC_dot.dot(O_r_PC_dot)
PD_squared = O_r_PD_dot.dot(O_r_PD_dot)
PE_squared = O_r_PE_dot.dot(O_r_PE_dot)

#Kinetic Energies

TA = 1/2 * m_rope * PA_squared
TB3 = 1/2 * m1 * PB3_squared
TC = 1/2 * m_rod * PC_squared
TD = 1/2 * m2 * PD_squared
TE = 1/2 * m3 * PE_squared

#Angular Velocities
A_ang_vel = A.ang_vel_in(N) #rope
B1_ang_vel = B1.ang_vel_in(N) #rolling wheel steering 
B2_ang_vel = B2.ang_vel_in(N) #rolling wheel tilting
B3_ang_vel = B3.ang_vel_in(N) #rolling wheel rolling

C_ang_vel = C.ang_vel_in(N) #rod
D_ang_vel = D.ang_vel_in(N) #balancing wheel
E_ang_vel = E.ang_vel_in(N) #steering wheel

#Inertias
I_A = 1/2 * m_rope * r_rope**2
I_B1 = 1/4 * m1 * r1**2
I_B2 = 1/4 * m1 * r1**2
I_B3 = 1/2 * m1 * r1**2
I_C = (1/12 * m_rod * (l1+l2)**2)* 2
I_D = 1/2 * m2 * r2**2 + 1/4 * m2 * r2**2 + 1/4 * m2 * r2**2 
I_E = 1/2 *m3 * r3**2 + 1/4 * m3 * r3**2 + 1/4 * m3 * r3**2 

# Rotational Kinetic Energies
T_rot_A = 0 #RotKE of rope
T_rot_B = 1/2 * (I_B1 * B1_ang_vel.dot(B1_ang_vel) + I_B2 * B2_ang_vel.dot(B2_ang_vel)+ I_B3 * B3_ang_vel.dot(B3_ang_vel)) #RotKE of rolling wheel
T_rot_C = 1/2 * I_C * C_ang_vel.dot(C_ang_vel) #RotKE of rod
T_rot_D = 1/2 * I_D * D_ang_vel.dot(D_ang_vel) #RotKE of balancing wheel
T_rot_E = 1/2 * I_E * E_ang_vel.dot(E_ang_vel) #RotKE of steering wheel


#Potential Energies
PE_PA = m_rope * g * O_r_PA.dot(N.z)
PE_PB3 = m1 * g * O_r_PB3.dot(N.z)
PE_PC = m_rod * g * O_r_PC.dot(N.z)
PE_PD = m2 * g * O_r_PD.dot(N.z)
PE_PE = m3 * g * O_r_PE.dot(N.z)

#Total Energies of the System
KE_system = TA + TB3 + TC + TD + TE + T_rot_A + T_rot_B + T_rot_C + T_rot_D + T_rot_E
PE_system = PE_PA + PE_PB3 + PE_PC + PE_PD + PE_PE

In [20]:
#Derivatives of KE wrt the degrees of freedom
dKE_dphi = KE_system.diff(phi)
dKE_dksi = KE_system.diff(ksi)
dKE_dtheta = KE_system.diff(theta)
dKE_ddelta = KE_system.diff(delta)
dKE_dgamma = KE_system.diff(gamma)
dKE_dbeta = KE_system.diff(beta)
dKE_dpsi =KE_system.diff(psi)

In [21]:
#Derivatives of KE wrt the derivatives of the degrees of freedom
dKE_dphi_dot = KE_system.diff(phi.diff(t)) #nonzero
dKE_dksi_dot = KE_system.diff(ksi.diff(t))
dKE_dtheta_dot = KE_system.diff(theta.diff(t)) #nonzero
dKE_ddelta_dot = KE_system.diff(delta.diff(t)) #nonzero
dKE_dgamma_dot = KE_system.diff(gamma.diff(t)) #nonzero
dKE_dbeta_dot = KE_system.diff(beta.diff(t)) #nonzero
dKE_dpsi_dot =KE_system.diff(psi.diff(t)) #nonzero

In [22]:
#Derivatives wrt time of the derivatives of KE wrt the derivatives of the degrees of freedom d(dT/dqdot)/dt
dKE_dphi_dot_dt = dKE_dphi_dot.diff(t) #nonzero long
dKE_dksi_dot_dt = dKE_dksi_dot.diff(t) #nonzero long
dKE_dpsi_dot_dt = dKE_dpsi_dot.diff(t) #0
dKE_dtheta_dot_dt = dKE_dtheta_dot.diff(t) #0
dKE_ddelta_dot_dt = dKE_ddelta_dot.diff(t) #nonzero long
dKE_dgamma_dot_dt = dKE_dgamma_dot.diff(t) #nonzero short
dKE_dbeta_dot_dt = dKE_dbeta_dot.diff(t) #0


In [23]:
#Derivatives of PE wrt the degrees of freedom
dPE_dphi = PE_system.diff(phi) #nonzero
dPE_dksi = PE_system.diff(ksi) #nonzero
dPE_dtheta = PE_system.diff(theta) #0
dPE_ddelta = PE_system.diff(delta) #nonzero
dPE_dgamma = PE_system.diff(gamma) #0
dPE_dbeta = PE_system.diff(beta) #0
dPE_dpsi =PE_system.diff(psi) #0

In [24]:
#Lagrange for each DOF
EoM_phi = dKE_dphi_dot_dt - dKE_dphi + dPE_dphi #nonzero and took 10 seconds to load
EoM_ksi = dKE_dksi_dot_dt - dKE_dksi+ dPE_dksi
EoM_theta = dKE_dtheta_dot_dt - dKE_dtheta+ dPE_dtheta #0
EoM_delta = dKE_ddelta_dot_dt - dKE_ddelta + dPE_ddelta #nonzero and horrible
EoM_gamma = dKE_dgamma_dot_dt - dKE_dgamma + dPE_dgamma #nonzero
EoM_beta = dKE_dbeta_dot_dt - dKE_dbeta + dPE_dbeta #0
EoM_psi = dKE_dpsi_dot_dt - dKE_dpsi + dPE_dpsi #0

In [ ]:
#EoM_phi
#EoM_ksi
#EoM_theta
#EoM_delta
#EoM_gamma
#EoM_beta
#EoM_psi

